# 1. Data collection

We collect electricity tariffs through the official ElCom data endpoint on LINDAS. The scope is the 2026 standard product for household profile H4, which assumes annual consumption of 4,500 kWh.

The notebook uses the saved API responses by default. To collect new snapshots, set `fetch_new_data = True`. Each new response is saved as received with a UTC timestamp.

In [1]:
# Install requests and pandas in the selected kernel if needed.
# %pip install requests pandas

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json

import pandas as pd
import requests

working_directory = Path.cwd()
project_directory = (
    working_directory.parent
    if working_directory.name == "notebooks"
    else working_directory
)
raw_directory = project_directory / "data" / "raw"

# Set to True to collect and save fresh snapshots from both APIs.
fetch_new_data = False

## 1.1 Load ElCom electricity tariffs

The SPARQL query selects municipality and operator URIs and the total tariff for the 2026 H4 standard product. A municipality can have several operators and therefore several tariff rows. Missing totals are retained for the data preparation checks.

In [3]:
# Official LINDAS SPARQL endpoint.
endpoint = "https://ld.admin.ch/query"

# Query for 2026 H4 standard-product tariffs.
query = """
PREFIX cube: <https://cube.link/>
PREFIX schema: <http://schema.org/>
PREFIX electricity: <https://energy.ld.admin.ch/elcom/electricityprice/dimension/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?municipality_uri ?operator_uri ?total_tariff

FROM <https://lindas.admin.ch/elcom/electricityprice>
FROM <https://lindas.admin.ch/territorial>
FROM <https://lindas.admin.ch/fso/register>

WHERE {
    <https://energy.ld.admin.ch/elcom/electricityprice>
        a cube:Cube ;
        cube:observationSet/cube:observation ?observation .

    ?observation
        electricity:period "2026"^^xsd:gYear ;
        electricity:category ?category ;
        electricity:product
            <https://energy.ld.admin.ch/elcom/electricityprice/product/standard> ;
        electricity:municipality ?municipality_uri ;
        electricity:operator ?operator_uri .

    ?category schema:name "H4" .

    OPTIONAL {
        ?observation electricity:total ?total_tariff .
    }
}

ORDER BY ?municipality_uri ?operator_uri
"""

In [4]:
if fetch_new_data:
    response = requests.post(
        endpoint,
        data={"query": query},
        headers={"Accept": "application/sparql-results+json"},
        timeout=120
    )
    response.raise_for_status()
    elcom_raw = response.json()
    retrieved_at = datetime.now(timezone.utc)
    print("HTTP status", response.status_code)
else:
    elcom_file = raw_directory / "elcom_2026_h4_20261117T195442966207Z.json"
    elcom_raw = json.loads(elcom_file.read_bytes())
    print("Loaded saved ElCom response", elcom_file.name)

print("Tariff rows", len(elcom_raw["results"]["bindings"]))

Loaded saved ElCom response elcom_2026_h4_20261117T195442966207Z.json
Tariff rows 2268


In [5]:
# Convert the nested response into table rows.
rows = [
    {field: entry["value"] for field, entry in record.items()}
    for record in elcom_raw["results"]["bindings"]
]

df_elcom = pd.DataFrame(rows, columns=elcom_raw["head"]["vars"]).rename(
    columns={
        "gemeinde": "municipality_uri",
        "betreiber": "operator_uri",
        "gesamttarif": "total_tariff"
    }
)

df_elcom.head()

,municipality_uri,operator_uri,total_tariff
0,https://ld.admin.ch/municipality/1,https://energy.ld.admin.ch/elcom/electricitypr...,24.136
1,https://ld.admin.ch/municipality/10,https://energy.ld.admin.ch/elcom/electricitypr...,24.136
2,https://ld.admin.ch/municipality/100,https://energy.ld.admin.ch/elcom/electricitypr...,24.136
3,https://ld.admin.ch/municipality/1001,https://energy.ld.admin.ch/elcom/electricitypr...,25.414
4,https://ld.admin.ch/municipality/1002,https://energy.ld.admin.ch/elcom/electricitypr...,25.414


## 1.2 Save ElCom raw data and retrieval metadata

When collecting fresh data, we save the server response as JSON and record the retrieval time, endpoint and exact query. The saved project snapshot retains the field names returned by the original query.

In [6]:
print("Raw data directory", raw_directory)

Raw data directory data/raw


In [7]:
if fetch_new_data:
    raw_directory.mkdir(parents=True, exist_ok=True)
    timestamp = retrieved_at.strftime("%Y%m%dT%H%M%S%fZ")
    elcom_file = raw_directory / f"elcom_2026_h4_{timestamp}.json"
    elcom_file.write_bytes(response.content)

    metadata = {
        "retrieved_at_utc": retrieved_at.isoformat(),
        "endpoint": endpoint,
        "tariff_year": 2026,
        "profile": "H4",
        "product": "standard",
        "tariff_row_count": len(elcom_raw["results"]["bindings"]),
        "raw_data_file": elcom_file.name,
        "sparql_query": query
    }
    metadata_file = raw_directory / f"{elcom_file.stem}_metadata.json"
    metadata_file.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print("Saved ElCom response and metadata", elcom_file.name, metadata_file.name)

## 1.3 Initial quality checks

We count tariff rows, distinct municipality and operator URIs, missing values, identical rows and duplicate municipality–operator pairs.

In [8]:
print("Tariff rows", len(df_elcom))
print("Distinct municipality URIs", df_elcom["municipality_uri"].nunique())
print("Distinct operator URIs", df_elcom["operator_uri"].nunique())
print("Missing values by column")
print(df_elcom.isna().sum())
print("Identical rows", df_elcom.duplicated().sum())
print(
    "Duplicate municipality–operator pairs",
    df_elcom.duplicated(subset=["municipality_uri", "operator_uri"]).sum()
)

Tariff rows 2268
Distinct municipality URIs 2122
Distinct operator URIs 551
Missing values by column
municipality_uri    0
operator_uri        0
total_tariff        1
dtype: int64
Identical rows 0
Duplicate municipality–operator pairs 0


### Results of the initial quality checks

The saved ElCom response contains 2,268 rows, 2,122 distinct municipality URIs and 551 distinct operator URIs. All municipality and operator identifiers are present. One row has a missing total tariff. There are zero identical rows and zero duplicate municipality–operator pairs.

Municipalities with multiple operators retain all their tariff rows. The data preparation notebook handles the missing total.

In [9]:
missing_tariffs = df_elcom[df_elcom["total_tariff"].isna()]

with pd.option_context("display.max_colwidth", None):
    display(missing_tariffs)

,municipality_uri,operator_uri,total_tariff
699,https://ld.admin.ch/municipality/3218,https://energy.ld.admin.ch/elcom/electricityprice/operator/326,NaN


## 1.4 Load BFS municipality data

We use the official BFS municipality register as of January 1, 2026. We select the municipality number, municipality name, canton identifier, canton name and 2020 urban–rural typology. The municipality number links these records to the ElCom tariffs.

In [10]:
bfs_url = (
    "https://www.agvchapp.bfs.admin.ch"
    "/api/communes/levels?date=01-01-2026"
)

if fetch_new_data:
    bfs_response = requests.get(bfs_url, timeout=120)
    bfs_response.raise_for_status()
    bfs_retrieved_at = datetime.now(timezone.utc)
    bfs_timestamp = bfs_retrieved_at.strftime("%Y%m%dT%H%M%S%fZ")
    bfs_file = raw_directory / f"bfs_municipalities_2026_{bfs_timestamp}.csv"
    bfs_file.write_bytes(bfs_response.content)
    print("HTTP status", bfs_response.status_code)
    print("Saved BFS response", bfs_file.name)
else:
    bfs_file = raw_directory / "bfs_municipalities_2026_20261117T195841888739Z.csv"
    print("Loaded saved BFS response", bfs_file.name)

Loaded saved BFS response bfs_municipalities_2026_20261117T195841888739Z.csv


In [11]:
df_bfs = pd.read_csv(bfs_file, encoding="utf-8-sig")

print("BFS rows", len(df_bfs))
print("Available columns")
print(df_bfs.columns.tolist())
df_bfs.head()

BFS rows 2110
Available columns
['HistoricalCode', 'BfsCode', 'Name', 'CantonId', 'Canton', 'DistrictId', 'District', 'FUA2014', 'DEGURB2021', 'FUA2021', 'GCITIES2021', 'STREG2022', 'DEGURB2011', 'GCITIES2011', 'REGCH', 'AGKG2020', 'STADTE2020', 'STGD2020', 'RSC2020', 'SPRGEB2020', 'AGKSA2020', 'AGGL2020', 'AGGLGK2020', 'MONT2019', 'STALAN2020', 'GDETYP2020_9', 'GDETYP2020_25', 'GBAE2018', 'BAE2018']


,HistoricalCode,BfsCode,Name,CantonId,Canton,DistrictId,District,FUA2014,DEGURB2021,FUA2021,...,SPRGEB2020,AGKSA2020,AGGL2020,AGGLGK2020,MONT2019,STALAN2020,GDETYP2020_9,GDETYP2020_25,GBAE2018,BAE2018
0,10078,6158,Vionnaz,23,Valais / Wallis,10013,District de Monthey,0,3,5586960,...,2,0,0,0,1,2,22,226,2,2040
1,10157,3023,Speicher,15,Appenzell Ausserrhoden,10098,Bezirk Mittelland,3203960,2,3203960,...,1,3203910,3203,3,1,2,21,217,15,15050
2,10159,6011,Zwischbergen,23,Valais / Wallis,10035,Bezirk Brig,0,3,0,...,1,0,0,0,1,3,33,338,7,7090
3,10162,2228,Villars-sur-Glâne,10,Fribourg / Freiburg,10104,District de la Sarine,0,1,2196960,...,2,2196910,2196,4,0,1,12,122,4,4030
4,10165,2230,Villarsel-sur-Marly,10,Fribourg / Freiburg,10104,District de la Sarine,0,3,2196960,...,2,2196910,2196,4,0,3,23,235,4,4030


In [12]:
if fetch_new_data:
    bfs_metadata = {
        "retrieved_at_utc": bfs_retrieved_at.isoformat(),
        "source": "Federal Statistical Office",
        "url": bfs_url,
        "municipality_reference_date": "2026-01-01",
        "typology": "Urban–rural typology 2020",
        "row_count": len(df_bfs),
        "columns": df_bfs.columns.tolist(),
        "raw_data_file": bfs_file.name
    }
    bfs_metadata_file = raw_directory / f"{bfs_file.stem}_metadata.json"
    bfs_metadata_file.write_text(
        json.dumps(bfs_metadata, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print("Saved BFS metadata", bfs_metadata_file.name)

In [13]:
df_bfs_selected = df_bfs[
    ["BfsCode", "Name", "CantonId", "Canton", "STALAN2020"]
].copy()

df_bfs_selected.head()

,BfsCode,Name,CantonId,Canton,STALAN2020
0,6158,Vionnaz,23,Valais / Wallis,2
1,3023,Speicher,15,Appenzell Ausserrhoden,2
2,6011,Zwischbergen,23,Valais / Wallis,3
3,2228,Villars-sur-Glâne,10,Fribourg / Freiburg,1
4,2230,Villarsel-sur-Marly,10,Fribourg / Freiburg,3


In [14]:
print("BFS municipalities", len(df_bfs_selected))
print("Missing values")
print(df_bfs_selected.isna().sum())
print("Duplicate municipality numbers", df_bfs_selected["BfsCode"].duplicated().sum())
print("Urban–rural code distribution")
print(df_bfs_selected["STALAN2020"].value_counts(dropna=False).sort_index())

BFS municipalities 2110
Missing values
BfsCode       0
Name          0
CantonId      0
Canton        0
STALAN2020    0
dtype: int64
Duplicate municipality numbers 0
Urban–rural code distribution
STALAN2020
1     509
2     580
3    1021
Name: count, dtype: int64
